BI-LSTM

In [1]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    SpatialDropout1D
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
x_train = joblib.load("/content/x_train.pkl")
x_test = joblib.load("/content/x_test.pkl")

y_train = joblib.load("/content/y_train.pkl")
y_test = joblib.load("/content/y_test.pkl")

Tokenizer

In [ ]:
VOCAB_SIZE = 50000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(x_train)

In [ ]:
x_tr_seq = tokenizer.texts_to_sequences(x_train)
x_te_seq = tokenizer.texts_to_sequences(x_test)

In [ ]:
seq_len = [len(seq) for seq in x_tr_seq]

print("Maximum Length :", max(seq_len))
print("Minimum Length :", min(seq_len))
print("Average Length :", np.mean(seq_len))
print("Median Length  :", np.median(seq_len))
print("95th Percentile:", np.percentile(seq_len, 95))

Maximum Length : 56
Minimum Length : 1
Average Length : 7.173818624180827
Median Length  : 7.0
95th Percentile: 14.0


Padding

In [ ]:
MAX_LENGTH = 20

x_tr_pad = pad_sequences(
    x_tr_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

x_te_pad = pad_sequences(
    x_te_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

In [ ]:
print(x_tr_pad.shape)
print(x_te_pad.shape)

(1274150, 20)
(318538, 20)


## Model

In [ ]:
VOCAB_SIZE = 50000
EMBEDDING_DIM = 128

model = Sequential([

    Input(shape=(MAX_LENGTH,)),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SpatialDropout1D(0.2),

    # Bidirectional(
    #     LSTM(
    #         256,
    #         return_sequences=True,
    #         dropout=0.2,
    #         recurrent_dropout=0.2
    #     )
    # ),

    Bidirectional(
        LSTM(
            128,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    ),

    Bidirectional(
        LSTM(
            64,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    ),

    Dense(128, activation="relu"),

    Dropout(0.4),

    Dense(64, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

In [ ]:
model.layers

[<Embedding name=embedding, built=True>,
 <SpatialDropout1D name=spatial_dropout1d, built=True>,
 <Bidirectional name=bidirectional, built=True>,
 <Bidirectional name=bidirectional_1, built=True>,
 <Dense name=dense, built=True>,
 <Dropout name=dropout, built=True>,
 <Dense name=dense_1, built=True>,
 <Dropout name=dropout_1, built=True>,
 <Dense name=dense_2, built=True>]

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 20, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 20, 256)        │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,852,353 (26.14 MB)

 Trainable params: 6,852,353 (26.14 MB)

 Non-trainable params: 0 (0.00 B)

Compile

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

Call Backs

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
start = time.time()

history = model.fit(
    x_tr_pad,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    callbacks=[
        early_stop,
        checkpoint,
        reduce_lr
    ],
    verbose=1
)
end = time.time()
print(f"Training Time: {(end-start)/60:.2f} Minutes")

Epoch 1/10
1991/1991 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step - accuracy: 0.7457 - loss: 0.5080 - precision: 0.7454 - recall: 0.7463
Epoch 1: val_accuracy improved from None to 0.79230, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
1991/1991 ━━━━━━━━━━━━━━━━━━━━ 559s 273ms/step - accuracy: 0.7743 - loss: 0.4743 - precision: 0.7738 - recall: 0.7751 - val_accuracy: 0.7923 - val_loss: 0.4418 - val_precision: 0.7898 - val_recall: 0.7969 - learning_rate: 0.0010
Epoch 2/10
1991/1991 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.8029 - loss: 0.4278 - precision: 0.8023 - recall: 0.8029
Epoch 2: val_accuracy improved from 0.79230 to 0.79612, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
1991/1991 ━━━━━━━━━━━━━━━━━━━━ 558s 280ms/step - accuracy: 0.8016 - loss: 0.4297 - precision: 0.8008 - recall: 0.8029 - val_accuracy: 0.7961 - val_loss: 0.4393 - val_precision: 0.7915 - val_recall: 0.8043 - learning_rate: 0.0010
Epoch 3/1

## Testing

In [ ]:
print(type(x_test))
print(type(y_test))

print(x_test.dtype)
print(y_test.dtype)

print(x_test.shape)
print(y_test.shape)

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
object
int64
(318538,)
(318538,)


In [ ]:
best_model = load_model("best_model.keras")

In [ ]:
test_loss, test_acc, test_precision, test_recall = best_model.evaluate(
    x_te_pad,
    y_test,
    verbose=1
)

print(f"Test Loss      : {test_loss:.4f}")
print(f"Test Accuracy  : {test_acc*100:.2f}%")
print(f"Test Precision : {test_precision:.4f}")
print(f"Test Recall    : {test_recall:.4f}")

9955/9955 ━━━━━━━━━━━━━━━━━━━━ 448s 45ms/step - accuracy: 0.7952 - loss: 0.4403 - precision: 0.7899 - recall: 0.8044
Test Loss      : 0.4403
Test Accuracy  : 79.52%
Test Precision : 0.7899
Test Recall    : 0.8044
